# VPN Config Checker — Results Analysis

Analyzes the CSV output from the VPN Config Checker tool.
Covers availability, protocol breakdown, geographic distribution, latency, and exit node characteristics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

results_dir = Path('../results/merged')
csv_files = sorted(results_dir.rglob('results.csv'))
assert csv_files, f'No results.csv found under {results_dir}'
CSV_PATH = csv_files[-1]
print(f'Loading: {CSV_PATH}')

pdf = PdfPages('report.pdf')
print('PDF report will be saved to: report.pdf')


def _pie_labels(ax, wedges, sizes, total, threshold=5):
    for wedge, val in zip(wedges, sizes):
        pct = val / total * 100
        if pct == 0:
            continue
        angle = np.radians((wedge.theta2 + wedge.theta1) / 2)
        x, y = np.cos(angle), np.sin(angle)
        if pct >= threshold:
            ax.text(0.6 * x, 0.6 * y, f'{pct:.1f}%',
                    ha='center', va='center', fontsize=12, fontweight='bold')
        else:
            ax.annotate(
                f'{pct:.1f}%',
                xy=(0.92 * x, 0.92 * y),
                xytext=(1.45 * x, 1.45 * y),
                ha='center', va='center', fontsize=11,
                arrowprops=dict(arrowstyle='->', color='gray', lw=1),
            )


def _save_text_page(pdf, content, title='', fontsize=8.5):
    """Render a text/table summary as a full PDF page."""
    if not isinstance(content, str):
        content = str(content)
    n_lines = content.count('\n') + 1
    if n_lines > 50:
        fontsize = 6.5
    elif n_lines > 35:
        fontsize = 7.5
    fig = plt.figure(figsize=(11, 8.5))
    fig.patch.set_facecolor('white')
    if title:
        fig.text(0.04, 0.97, title, fontsize=13, fontweight='bold', va='top', color='#1A237E')
        fig.text(0.04, 0.91, content, fontsize=fontsize, va='top',
                 family='monospace', color='#212121', linespacing=1.4)
    else:
        fig.text(0.04, 0.97, content, fontsize=fontsize, va='top',
                 family='monospace', color='#212121', linespacing=1.4)
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

In [ ]:

# Save every plot as a PNG in analysis_plots/ alongside this notebook
_plots_dir = Path('analysis_plots')
_plots_dir.mkdir(exist_ok=True)
_plot_counter = [0]
_orig_plt_show = plt.show

def _autosave_show(*args, **kwargs):
    _plot_counter[0] += 1
    plt.gcf().savefig(
        _plots_dir / f'plot_{_plot_counter[0]:02d}.png',
        bbox_inches='tight', dpi=150,
    )
    _orig_plt_show(*args, **kwargs)

plt.show = _autosave_show
print(f'Plots will be auto-saved to: {_plots_dir.resolve()}')


In [ ]:
df = pd.read_csv(CSV_PATH)

df['is_redirecting'] = df['is_redirecting'].fillna(False).astype(bool)
df['is_datacenter']  = df['is_datacenter'].fillna(False).astype(bool)
df['is_blacklisted'] = df['is_blacklisted'].fillna(False).astype(bool)

for col in ['proxy_detected', 'ipv6_leak', 'dns_leak']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().eq('true')

working = df[df['is_redirecting']].copy()

print(df.dtypes)
df.head(3)

---
## 1. High-Level Counts

In [ ]:
total       = len(df)
available   = df['is_redirecting'].sum()
unavailable = total - available
datacenter  = working['is_datacenter'].sum()
blacklisted = working['is_blacklisted'].sum()

print('=' * 45)
print(f'  Total configs tested   : {total:>6}')
print('─' * 45)
print(f'  Available (redirecting): {available:>6}  ({available/total*100:.1f}%)')
print(f'  Unavailable            : {unavailable:>6}  ({unavailable/total*100:.1f}%)')
print('─' * 45)
print(f'  Datacenter exits       : {datacenter:>6}  ({datacenter/available*100:.1f}% of working)')
print(f'  Blacklisted exits      : {blacklisted:>6}  ({blacklisted/available*100:.1f}% of working)')
print('=' * 45)

_save_text_page(pdf, (
    f"Total configs tested   : {total:,}\n"
    f"{'─'*45}\n"
    f"Available (redirecting): {available:,}  ({available/total*100:.1f}%)\n"
    f"Unavailable            : {unavailable:,}  ({unavailable/total*100:.1f}%)\n"
    f"{'─'*45}\n"
    f"Datacenter exits       : {datacenter:,}  ({datacenter/available*100:.1f}% of working)\n"
    f"Blacklisted exits      : {blacklisted:,}  ({blacklisted/available*100:.1f}% of working)"
), title='1. High-Level Config Overview')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
labels = ['Available', 'Unavailable']
sizes  = [available, unavailable]
colors = ['#4CAF50', '#F44336']
wedges, _ = ax.pie(
    sizes, colors=colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
)
_pie_labels(ax, wedges, sizes, total)
ax.legend(wedges, labels, loc='lower center', bbox_to_anchor=(0.5, -0.08), fontsize=11)
ax.set_title('Availability', fontsize=14, fontweight='bold')

ax2 = axes[1]
labels2 = ['Datacenter / Hosting', 'Residential / Business']
sizes2  = [int(datacenter), int(available - datacenter)]
colors2 = ['#5C6BC0', '#FF7043']
wedges2, _ = ax2.pie(
    sizes2, colors=colors2, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
)
_pie_labels(ax2, wedges2, sizes2, int(available))
ax2.legend(wedges2, labels2, loc='lower center', bbox_to_anchor=(0.5, -0.08), fontsize=11)
ax2.set_title(f'Exit Node Type — Working Configs\n(n={int(available):,})', fontsize=13, fontweight='bold')

plt.suptitle(f'Config Overview  (n={total:,})', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 2. Protocol Breakdown

In [ ]:
proto = df.groupby('protocol').agg(
    total     = ('protocol', 'count'),
    available = ('is_redirecting', 'sum'),
).reset_index()
proto['avail_%'] = (proto['available'] / proto['total'] * 100).round(1)
proto.sort_values('total', ascending=False, inplace=True)

_save_text_page(pdf, proto.to_string(index=False), title='2. Protocol Availability Table')

proto

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
x = range(len(proto))
w = 0.35
b1 = ax.bar([i - w/2 for i in x], proto['total'],     width=w, label='Total',     color='#90A4AE')
b2 = ax.bar([i + w/2 for i in x], proto['available'], width=w, label='Available', color='#4CAF50')
ax.set_xticks(list(x))
ax.set_xticklabels(proto['protocol'].str.upper(), fontsize=11)
ax.set_ylabel('Number of configs')
ax.set_title('Protocol — Total vs Available (count)', fontsize=12, fontweight='bold')
ax.legend()
for rect in b1:
    h = rect.get_height()
    if h > 0:
        ax.text(rect.get_x() + rect.get_width()/2, h * 1.01, str(int(h)),
                ha='center', va='bottom', fontsize=8)
for rect in b2:
    h = rect.get_height()
    if h > 0:
        ax.text(rect.get_x() + rect.get_width()/2, h * 1.01, str(int(h)),
                ha='center', va='bottom', fontsize=9, color='#1B5E20', fontweight='bold')

ax2 = axes[1]
rate_colors = ['#1B5E20' if r >= 10 else '#66BB6A' if r >= 2 else '#FFA726' if r >= 0.5 else '#EF5350'
               for r in proto['avail_%']]
bars = ax2.bar(proto['protocol'].str.upper(), proto['avail_%'],
               color=rate_colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, proto['avail_%']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylabel('Availability Rate (%)')
ax2.set_title('Protocol — Availability Rate\n(dark green ≥ 10%  |  amber 0.5–2%  |  red < 0.5%)',
              fontsize=12, fontweight='bold')
ax2.set_ylim(0, proto['avail_%'].max() * 1.35)

plt.suptitle('Protocol Breakdown', fontsize=14, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 2b. Latency by Exit Node Type

Compare latency distributions between datacenter and residential exits.

In [ ]:
lat_dc  = working[working['is_datacenter']]['latency_ms'].dropna()
lat_res = working[~working['is_datacenter']]['latency_ms'].dropna()

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(lat_dc.clip(upper=5000),  bins=40, alpha=0.6, color='#5C6BC0',
        label=f'Datacenter (n={len(lat_dc)})', edgecolor='white')
ax.hist(lat_res.clip(upper=5000), bins=40, alpha=0.6, color='#FF7043',
        label=f'Residential (n={len(lat_res)})', edgecolor='white')
if len(lat_dc):
    ax.axvline(lat_dc.median(),  color='#3949AB', linestyle='--', linewidth=1.5,
               label=f'DC median {lat_dc.median():.0f}ms')
if len(lat_res):
    ax.axvline(lat_res.median(), color='#E64A19', linestyle='--', linewidth=1.5,
               label=f'Residential median {lat_res.median():.0f}ms')
ax.set_xlabel('Latency (ms, capped at 5000)')
ax.set_ylabel('Config count')
ax.set_title('Latency Distribution — Datacenter vs Residential Exits',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

if len(lat_dc):
    print(f'Datacenter  — median: {lat_dc.median():.0f}ms   mean: {lat_dc.mean():.0f}ms   n={len(lat_dc)}')
if len(lat_res):
    print(f'Residential — median: {lat_res.median():.0f}ms   mean: {lat_res.mean():.0f}ms   n={len(lat_res)}')

---
## 2c. Exit Node Breakdown — Datacenter & Blacklisted IPs

How many working configs exit through datacenter ranges or known proxy/VPN IPs (ip-api.com flags).

In [ ]:
flag_counts = pd.Series({
    'Datacenter / Hosting':  int(working['is_datacenter'].sum()),
    'Blacklisted / Proxy IP': int(working['is_blacklisted'].sum()),
    'Residential exit':       int((~working['is_datacenter']).sum()),
}).sort_values(ascending=False)

flag_pct = (flag_counts / len(working) * 100).round(1)

print(f'Working configs analysed: {len(working)}\n')
print(pd.DataFrame({'count': flag_counts, 'pct_of_working': flag_pct}).to_string())

colors = ['#5C6BC0', '#E53935', '#FF7043']
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(flag_counts.index[::-1], flag_counts.values[::-1],
               color=colors[:len(flag_counts)][::-1], edgecolor='white')
for bar, val, p in zip(bars, flag_counts.values[::-1], flag_pct.values[::-1]):
    ax.text(
        bar.get_width() + max(flag_counts) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val}  ({p}%)', va='center', fontsize=10,
    )
ax.set_xlabel('Number of working configs')
ax.set_title('Exit Node Characteristics\n(among all redirecting / working configs)',
             fontsize=13, fontweight='bold')
ax.set_xlim(right=flag_counts.max() * 1.25)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 3. Error / Failure Reasons

In [ ]:
failed = df[~df['is_redirecting']].copy()
error_counts = failed['error'].fillna('Unknown').value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(error_counts.index[::-1], error_counts.values[::-1],
               color='#EF9A9A', edgecolor='white')
for bar, val in zip(bars, error_counts.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)
ax.set_xlabel('Number of configs')
ax.set_title('Top Failure Reasons (non-redirecting configs)', fontsize=13, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

print(error_counts.to_string())
_save_text_page(pdf, error_counts.to_string(), title='3. Top Failure Reasons (non-redirecting configs)')

---
## 4. Geographic Distribution of Exit Nodes

In [ ]:
working = df[df['is_redirecting']].copy()

country_counts = working['country'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(country_counts.index[::-1], country_counts.values[::-1],
               color=sns.color_palette('Blues_r', len(country_counts)))
for bar, val in zip(bars, country_counts.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)
ax.set_xlabel('Number of working configs')
ax.set_title('Top 20 Exit Countries', fontsize=13, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

In [ ]:
top15 = country_counts.head(15).index.tolist()
geo = working[working['country'].isin(top15)].groupby('country').agg(
    count           = ('protocol', 'count'),
    avg_latency     = ('latency_ms', 'mean'),
    datacenter_rate = ('is_datacenter', 'mean'),
).loc[top15]
geo['datacenter_pct'] = (geo['datacenter_rate'] * 100).round(1)
geo = geo.sort_values('avg_latency')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
bars = ax.barh(geo.index, geo['avg_latency'],
               color=sns.color_palette('Blues_r', len(geo)), edgecolor='white')
for bar, val in zip(bars, geo['avg_latency']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}ms', va='center', fontsize=9)
ax.set_xlabel('Average Latency (ms)')
ax.set_title('Avg Latency — Top 15 Exit Countries', fontsize=13, fontweight='bold')

ax2 = axes[1]
bars2 = ax2.barh(geo.index, geo['datacenter_pct'],
                 color='#5C6BC0', edgecolor='white')
for bar, val in zip(bars2, geo['datacenter_pct']):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.0f}%', va='center', fontsize=9)
ax2.set_xlabel('Datacenter Exit Rate (%)')
ax2.set_title('Datacenter Exit Rate — Top 15 Exit Countries', fontsize=13, fontweight='bold')

plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 5. Top ISPs / Organizations

In [ ]:
org_counts = working['org'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(org_counts.index[::-1], org_counts.values[::-1],
               color=sns.color_palette('Greens_r', len(org_counts)))
for bar, val in zip(bars, org_counts.values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10)
ax.set_xlabel('Number of working configs')
ax.set_title('Top 15 Exit Node Organizations (ISP / Hosting)', fontsize=13, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 6. Exit IP Risk Profile — Datacenter × Blacklist Cross-Reference

Four-way breakdown: configs are either datacenter or residential, and either blacklisted or clean.
Also compares median latency across the four categories.

In [ ]:
working['exit_type'] = working.apply(lambda r:
    'DC + Blacklisted'   if r['is_datacenter'] and r['is_blacklisted'] else
    'DC only'            if r['is_datacenter'] else
    'Res + Blacklisted'  if r['is_blacklisted'] else
    'Clean Residential', axis=1)

ordered_types = ['DC + Blacklisted', 'DC only', 'Res + Blacklisted', 'Clean Residential']
type_counts   = working['exit_type'].value_counts()
type_lat      = working.groupby('exit_type')['latency_ms'].median()

print('Exit IP risk breakdown (working configs):')
for t in ordered_types:
    cnt = type_counts.get(t, 0)
    pct = cnt / len(working) * 100
    med = type_lat.get(t, float('nan'))
    print(f'  {t:<22}  {cnt:>4}  ({pct:.1f}%)  median latency: {med:.0f}ms')

_save_text_page(pdf, (
    'Exit IP risk breakdown (working configs):\n' +
    '\n'.join(
        f'  {t:<22}  {type_counts.get(t, 0):>4}  ({type_counts.get(t, 0)/len(working)*100:.1f}%)'
        f'  median latency: {type_lat.get(t, float("nan")):.0f}ms'
        for t in ordered_types
    )
), title='6. Exit IP Risk Profile — Datacenter × Blacklist')

colors_type = {
    'DC + Blacklisted':  '#B71C1C',
    'DC only':           '#5C6BC0',
    'Res + Blacklisted': '#FF7043',
    'Clean Residential': '#43A047',
}
counts_ordered = [type_counts.get(t, 0) for t in ordered_types]
lats_ordered   = [type_lat.get(t, float('nan')) for t in ordered_types]
colors_ordered = [colors_type[t] for t in ordered_types]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.bar(ordered_types, counts_ordered, color=colors_ordered, edgecolor='white', width=0.6)
for bar, cnt in zip(bars, counts_ordered):
    pct = cnt / len(working) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of working configs')
ax.set_title('Exit IP Profile — 4 Risk Categories', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(counts_ordered) * 1.3)
ax.set_xticklabels(ordered_types, rotation=15, ha='right')

ax2 = axes[1]
bars2 = ax2.bar(ordered_types, lats_ordered, color=colors_ordered, edgecolor='white', width=0.6)
for bar, val in zip(bars2, lats_ordered):
    if not np.isnan(val):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:.0f}ms', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylabel('Median Latency (ms)')
ax2.set_title('Median Latency by Risk Category', fontsize=13, fontweight='bold')
ax2.set_xticklabels(ordered_types, rotation=15, ha='right')

plt.suptitle('Exit IP Risk Profile — Datacenter × Blacklist', fontsize=14, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 7. Latency Distribution

In [ ]:
lat = working['latency_ms'].dropna()

print(f'Count  : {len(lat)}')
print(f'Mean   : {lat.mean():.0f} ms')
print(f'Median : {lat.median():.0f} ms')
print(f'Min    : {lat.min():.0f} ms')
print(f'Max    : {lat.max():.0f} ms')
print(f'Std    : {lat.std():.0f} ms')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(lat.clip(upper=5000), bins=50, color='#42A5F5', edgecolor='white')
ax.axvline(lat.median(), color='red', linestyle='--', label=f'Median {lat.median():.0f}ms')
ax.axvline(lat.mean(),   color='orange', linestyle='--', label=f'Mean {lat.mean():.0f}ms')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Config count')
ax.set_title('Latency Distribution (capped at 5000ms)', fontweight='bold')
ax.legend()

ax2 = axes[1]
proto_lat = working[['protocol', 'latency_ms']].dropna()
order = proto_lat.groupby('protocol')['latency_ms'].median().sort_values().index
sns.boxplot(data=proto_lat, x='protocol', y='latency_ms', order=order,
            palette='muted', ax=ax2, showfliers=False)
ax2.set_xlabel('Protocol')
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Latency by Protocol (outliers hidden)', fontweight='bold')

plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 8. Latency Buckets

In [ ]:
bins   = [0, 300, 600, 1000, 2000, 5000, float('inf')]
labels_b = ['<300ms', '300–600ms', '600ms–1s', '1–2s', '2–5s', '>5s']
working_copy = working.copy()
working_copy['latency_bucket'] = pd.cut(working_copy['latency_ms'], bins=bins, labels=labels_b)
bucket_counts = working_copy['latency_bucket'].value_counts().reindex(labels_b, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(bucket_counts.index, bucket_counts.values,
              color=['#1B5E20','#388E3C','#66BB6A','#FFA726','#EF5350','#B71C1C'])
for bar, val in zip(bars, bucket_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=11)
ax.set_xlabel('Latency range')
ax.set_ylabel('Number of configs')
ax.set_title('Working Configs by Latency Bucket', fontsize=13, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

---
## 9. Config Source & Telegram Channel Analysis

Parses the `name` field to identify Telegram channels and other config sources.
Compares config quality (latency, datacenter rate) across top channels.

In [ ]:
import re

wc = working.copy()
wc['channel'] = wc['name'].str.extract(r'(@[\w\d_]+)', expand=False).str.lower()

channel_counts = wc['channel'].value_counts()
top_channels   = channel_counts.head(15)

print(f'Working configs total         : {len(wc)}')
print(f'Working configs with @channel : {wc["channel"].notna().sum()}  ({wc["channel"].notna().mean()*100:.1f}%)')
print(f'Unique @channels              : {wc["channel"].nunique()}')
print()
print('Top channels by working config count:')
print(top_channels.to_string())

_save_text_page(pdf, (
    f"Working configs total         : {len(wc)}\n"
    f"Working configs with @channel : {wc['channel'].notna().sum()}  ({wc['channel'].notna().mean()*100:.1f}%)\n"
    f"Unique @channels              : {wc['channel'].nunique()}\n\n"
    f"Top channels by working config count:\n{top_channels.to_string()}"
), title='9. Telegram Channel Analysis')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
if len(top_channels) > 0:
    bars = ax.barh(top_channels.index[::-1], top_channels.values[::-1],
                   color=sns.color_palette('Purples_r', len(top_channels)), edgecolor='white')
    for bar, val in zip(bars, top_channels.values[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10)
    ax.set_xlabel('Number of working configs')
    ax.set_title('Top 15 Telegram Channels\n(by working config count)', fontsize=12, fontweight='bold')
    ax.set_xlim(right=top_channels.max() * 1.2)

ax2 = axes[1]
name_breakdown = pd.Series({
    'Named @channel': int(wc['channel'].notna().sum()),
    'Named (other)':  int((wc['name'].notna() & wc['channel'].isna()).sum()),
})
if name_breakdown.sum() > 0:
    name_breakdown.plot(kind='pie', ax=ax2,
                        colors=['#7B1FA2', '#CE93D8'],
                        autopct='%1.1f%%', startangle=90,
                        wedgeprops=dict(edgecolor='white', linewidth=2),
                        textprops={'fontsize': 12})
ax2.set_title('Config Attribution\n(all working configs have a name field)',
              fontsize=12, fontweight='bold')
ax2.set_ylabel('')

plt.suptitle('Config Sources — Telegram Channel Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

if len(top_channels) >= 3:
    top5     = top_channels.head(5).index.tolist()
    ch_stats = (
        wc[wc['channel'].isin(top5)]
        .groupby('channel')
        .agg(
            count           = ('latency_ms', 'count'),
            median_lat_ms   = ('latency_ms', 'median'),
            pct_datacenter  = ('is_datacenter', 'mean'),
            pct_blacklisted = ('is_blacklisted', 'mean'),
        )
    )
    ch_stats['pct_datacenter']  = (ch_stats['pct_datacenter']  * 100).round(1)
    ch_stats['pct_blacklisted'] = (ch_stats['pct_blacklisted'] * 100).round(1)
    ch_stats.sort_values('median_lat_ms', inplace=True)
    print('\nTop-channel quality comparison (working configs):')
    print(ch_stats.to_string())
    _save_text_page(pdf, ch_stats.to_string(), title='9b. Top-Channel Quality Comparison')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ch_colors = sns.color_palette('Purples_r', len(ch_stats))

    ax = axes[0]
    bars = ax.barh(ch_stats.index, ch_stats['median_lat_ms'],
                   color=ch_colors, edgecolor='white')
    for bar, val in zip(bars, ch_stats['median_lat_ms']):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{val:.0f}ms', va='center', fontsize=10)
    ax.set_xlabel('Median Latency (ms)')
    ax.set_title('Top Channels — Median Latency', fontsize=12, fontweight='bold')

    ax2 = axes[1]
    bars2 = ax2.barh(ch_stats.index, ch_stats['pct_datacenter'],
                     color=ch_colors, edgecolor='white')
    for bar, val in zip(bars2, ch_stats['pct_datacenter']):
        ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}%', va='center', fontsize=10)
    ax2.set_xlabel('Datacenter Exit Rate (%)')
    ax2.set_title('Top Channels — Datacenter Exit Rate', fontsize=12, fontweight='bold')

    plt.suptitle('Top Telegram Channel Config Quality Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()

---
## 10. Full Summary Table

In [ ]:
summary = {
    'Total configs tested':           total,
    'Available (redirecting)':        int(available),
    'Unavailable':                    int(unavailable),
    'Availability rate':              f'{available/total*100:.1f}%',
    '─────────────────────────────':  '─────',
    'Datacenter exits':               int(datacenter),
    'Residential exits':              int(available - datacenter),
    'Blacklisted exits':              int(blacklisted),
    'Datacenter rate (of working)':   f'{datacenter/available*100:.1f}%' if available else 'N/A',
    '──────────────────────────────': '─────',
    'Unique exit IPs':                int(working['exit_ip'].nunique()),
    'Unique exit countries':          int(working['country'].nunique()),
    'Avg latency (working)':          f"{working['latency_ms'].mean():.0f} ms",
    'Median latency (working)':       f"{working['latency_ms'].median():.0f} ms",
    'Fastest config':                 f"{working['latency_ms'].min():.0f} ms",
}

for k, v in summary.items():
    print(f'  {k:<38} {v}')

_save_text_page(pdf,
    '\n'.join(f'  {k:<38} {v}' for k, v in summary.items()),
    title='10. Full Summary Table')

---
## 11. Security Profile

New fields collected during live proxy connection: proxy detection, IPv6 support, IPv6 leak detection.

**Note on empty fields:** `dns_resolver_ip` and `x_forwarded_for` are empty for all proxies because SOCKS5 operates at layer 4 — DNS resolves server-side (no EDNS client subnet) and HTTP headers are not modified.

In [ ]:
working['proxy_detected'] = working['proxy_detected'].astype(str).str.lower().eq('true') if 'proxy_detected' in working.columns else False
working['ipv6_leak']      = working['ipv6_leak'].astype(str).str.lower().eq('true') if 'ipv6_leak' in working.columns else False
working['has_ipv6']       = working['ipv6_exit_ip'].notna() if 'ipv6_exit_ip' in working.columns else False

n           = len(working)
proxy_det_n = int(working['proxy_detected'].sum())
ipv6_n      = int(working['has_ipv6'].sum())
ipv6_leak_n = int(working['ipv6_leak'].sum())

print(f'Working configs              : {n}')
print(f'Proxy-detected by ip-api.com : {proxy_det_n}  ({proxy_det_n/n*100:.1f}%)')
print(f'IPv6-capable exits           : {ipv6_n}  ({ipv6_n/n*100:.1f}%)')
print(f'IPv6 leak (country mismatch) : {ipv6_leak_n}  ({ipv6_leak_n/max(ipv6_n,1)*100:.1f}% of IPv6-capable)')
print()
print('dns_resolver_ip : empty — SOCKS5 routes DNS server-side; edns.ip-api.com requires')
print('                  EDNS client subnet, which most VPN proxy servers do not send.')
print('x_forwarded_for : empty — SOCKS5 is layer-4 and does not modify HTTP headers.')

_save_text_page(pdf, (
    f"Working configs              : {n}\n"
    f"Proxy-detected by ip-api.com : {proxy_det_n}  ({proxy_det_n/n*100:.1f}%)\n"
    f"IPv6-capable exits           : {ipv6_n}  ({ipv6_n/n*100:.1f}%)\n"
    f"IPv6 leak (country mismatch) : {ipv6_leak_n}  ({ipv6_leak_n/max(ipv6_n,1)*100:.1f}% of IPv6-capable)\n\n"
    f"dns_resolver_ip : empty — SOCKS5 routes DNS server-side; edns.ip-api.com requires\n"
    f"                  EDNS client subnet, which most VPN proxy servers do not send.\n"
    f"x_forwarded_for : empty — SOCKS5 is layer-4 and does not modify HTTP headers."
), title='11. Security Profile — Summary Stats')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
pd.Series({'Proxy-detected': proxy_det_n, 'Not detected': n - proxy_det_n}).plot(
    kind='pie', ax=ax, colors=['#E53935', '#4CAF50'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2), textprops={'fontsize': 10})
ax.set_title('Proxy Detection\n(ip-api.com queried through proxy)', fontsize=11, fontweight='bold')
ax.set_ylabel('')

ax2 = axes[1]
pd.Series({'IPv6 capable': ipv6_n, 'IPv4 only': n - ipv6_n}).plot(
    kind='pie', ax=ax2, colors=['#1976D2', '#90A4AE'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2), textprops={'fontsize': 10})
ax2.set_title('IPv6 Support', fontsize=11, fontweight='bold')
ax2.set_ylabel('')

ax3 = axes[2]
pd.Series({'IPv6 leak': ipv6_leak_n, 'IPv6 OK': max(ipv6_n - ipv6_leak_n, 0), 'IPv4 only': n - ipv6_n}).plot(
    kind='pie', ax=ax3, colors=['#F57C00', '#43A047', '#90A4AE'],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2), textprops={'fontsize': 10})
ax3.set_title('IPv6 Leak\n(exit country differs from IPv4)', fontsize=11, fontweight='bold')
ax3.set_ylabel('')

plt.suptitle('Security Profile — Working Configs', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

proto_sec = working.groupby('protocol').agg(
    count     = ('protocol', 'count'),
    proxy_pct = ('proxy_detected', 'mean'),
    ipv6_pct  = ('has_ipv6', 'mean'),
).reset_index()
proto_sec['proxy_pct'] = (proto_sec['proxy_pct'] * 100).round(1)
proto_sec['ipv6_pct']  = (proto_sec['ipv6_pct']  * 100).round(1)
proto_sec.sort_values('proxy_pct', ascending=False, inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.bar(proto_sec['protocol'].str.upper(), proto_sec['proxy_pct'],
              color=sns.color_palette('Reds_r', len(proto_sec)), edgecolor='white')
for bar, val in zip(bars, proto_sec['proxy_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}%', ha='center', fontsize=10)
ax.set_xlabel('Protocol')
ax.set_ylabel('Proxy-detected rate (%)')
ax.set_title('Proxy Detection Rate by Protocol\n(high = exit IP is in ip-api.com proxy DB)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 110)

ax2 = axes[1]
proto_ipv6 = proto_sec.sort_values('ipv6_pct', ascending=False)
bars2 = ax2.bar(proto_ipv6['protocol'].str.upper(), proto_ipv6['ipv6_pct'],
                color=sns.color_palette('Blues_r', len(proto_ipv6)), edgecolor='white')
for bar, val in zip(bars2, proto_ipv6['ipv6_pct']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', fontsize=10)
ax2.set_xlabel('Protocol')
ax2.set_ylabel('IPv6-capable rate (%)')
ax2.set_title('IPv6 Support Rate by Protocol', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 110)

plt.suptitle('Security — Protocol Breakdown', fontsize=13, fontweight='bold')
plt.tight_layout()
pdf.savefig(fig, bbox_inches='tight')
plt.show()

sec_by_type = working.groupby('is_datacenter').agg(
    count     = ('protocol', 'count'),
    proxy_pct = ('proxy_detected', 'mean'),
    ipv6_pct  = ('has_ipv6', 'mean'),
    bl_pct    = ('is_blacklisted', 'mean'),
).rename(index={True: 'Datacenter', False: 'Residential'})
sec_by_type[['proxy_pct', 'ipv6_pct', 'bl_pct']] *= 100

print('\nSecurity metrics by exit type:')
print(f'  {"Type":<15}  {"Count":>6}  {"Proxy-det%":>10}  {"IPv6%":>7}  {"Blacklist%":>10}')
for idx, row in sec_by_type.iterrows():
    print(f'  {idx:<15}  {int(row["count"]):>6}  {row["proxy_pct"]:>10.1f}  {row["ipv6_pct"]:>7.1f}  {row["bl_pct"]:>10.1f}')

_save_text_page(pdf, (
    f'  {"Type":<15}  {"Count":>6}  {"Proxy-det%":>10}  {"IPv6%":>7}  {"Blacklist%":>10}\n' +
    '\n'.join(
        f'  {idx:<15}  {int(row["count"]):>6}  {row["proxy_pct"]:>10.1f}  {row["ipv6_pct"]:>7.1f}  {row["bl_pct"]:>10.1f}'
        for idx, row in sec_by_type.iterrows()
    )
), title='11b. Security Metrics by Exit Type')

if ipv6_n > 0:
    ipv6_countries = working[working['has_ipv6']]['country'].value_counts().head(10)
    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.barh(ipv6_countries.index[::-1], ipv6_countries.values[::-1],
                   color='#1976D2', edgecolor='white')
    for bar, val in zip(bars, ipv6_countries.values[::-1]):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10)
    ax.set_xlabel('Number of IPv6-capable configs')
    ax.set_title('Top 10 Countries with IPv6-capable Exit Nodes', fontsize=13, fontweight='bold')
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.show()

---
## 12. Export: Working Configs Only

In [ ]:
export_cols = [
    'config_id', 'protocol', 'server', 'port', 'name',
    'exit_ip', 'country', 'country_code', 'city', 'org', 'asn',
    'is_datacenter', 'is_blacklisted', 'latency_ms', 'source',
]
sec_export_cols = ['proxy_detected', 'has_ipv6', 'ipv6_leak', 'ipv6_exit_ip']
for c in sec_export_cols:
    if c in working.columns:
        export_cols.append(c)

working_out = working[export_cols].sort_values('latency_ms')

out_path = Path('working_configs.csv')
working_out.to_csv(out_path, index=False)
print(f'Saved {len(working_out)} working configs → {out_path}')
working_out.head(10)

In [ ]:
pdf.close()
print('Saved → report.pdf')